# 23 plot pick event diagnostics

23_plot_pick_event_diagnostics.py

STEP 23 of the Pinatubo FAIR pipeline.

Plot and export diagnostics for:
- STEP 20: individual PHA picks
- STEP 21: monthly PHA picks
- STEP 22: merged picks

Produces:
- timeseries plots: picks/day, events/day, median picks/event/day, median duration/day
- distribution plots: picks/event, duration, P-S delay
- station/phase health plots
- CSV summary tables
- optional QC flags JSON

Assumptions:
- Input tables contain at least: event_id, pick_time
- pick_time is parseable by pandas.to_datetime(..., utc=True)

## Scope and execution

Dataset paths, source priority, tolerances and output naming stay in this Pinatubo workflow.

Review `DATA_TOP` and `SETTINGS`, then run the processing cell. Outputs may overwrite existing products according to the original step. The implementation is in the adjacent local module; no command-line script or subprocess is invoked.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / 'pipeline', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths
from pinatubo_steps.step_23_plot_pick_event_diagnostics import run

DATA_TOP = Path.home() / 'Dropbox/PROFESSIONAL/DATA/Pinatubo/ASSEMBLED-25-028-THOMPSON-PINATUBO1991'
P = pipeline_paths(DATA_TOP)

SETTINGS = dict(
    individual=P['INDIV_PICK_INDEX'],
    monthly=P['MONTHLY_PICK_INDEX'],
    merged=P['MERGED_PICK_INDEX'],
    outdir=P['STEP23_DIR'],
    top_stations=int(P['MAX_STATION_COUNT']),
    ps_delay_max=float(P['MAX_PS_DELAY']),
    emit_csv=True,
    emit_qc=True,
)
display(SETTINGS)


## Run the step

Imports and settings above do not run the analysis. This cell writes the configured products.


In [ ]:
result = run(**SETTINGS)


In [ ]:
# Preview returned tables and summary objects without reloading output files.
for name, value in result.items():
    print(name)
    display(value.head() if hasattr(value, 'head') else
            value[:5] if isinstance(value, list) else value)


In [ ]:
from IPython.display import Image

# Show saved diagnostics inline; all figures remain in the configured folder.
plot_roots = [Path(SETTINGS[k]) for k in ('outdir', 'out_dir', 'out')
              if k in SETTINGS and SETTINGS[k] is not None]
plots = sorted({p for root in plot_roots
                for p in ([root] if root.suffix.lower() == '.png'
                          else root.rglob('*.png')) if p.is_file()})
print(f'{len(plots)} saved PNG figures; showing up to four.')
for plot in plots[:4]:
    print(plot)
    display(Image(filename=str(plot)))


## Individual ↔ monthly PHA cross-check

This section can run independently after the common setup cell; it does not require step 22 or rerun either parser. It reads existing step-20/21 CSVs and writes separate reports. No picks are reassigned, suppressed or rewritten.

Match identity is the full normalized SEED ID and phase, including network, location, channel orientation and gain. Each tolerance is inclusive. Overlap means at least one counterpart exists, not one-to-one agreement. Mutual-unique pairs have exactly one candidate on both sides; multiple candidates and reused counterparts are reported explicitly. Signed offsets are monthly minus individual time.

Dates select UTC reporting days, while candidate matching retains neighboring-day context across midnight. Empty denominators produce NaN, not a zero success rate. Missing input files raise an error. Invalid records remain in their own audit table; rows with unknown times cannot be attributed to a selected day.

Date audits compare folder, filename and pick dates only where metadata permits. Future step-20 outputs include pha_path. Older CSVs containing only a basename cannot support a folder-date check; the report marks insufficient metadata rather than guessing. Source agreement alone does not prove parser correctness.


In [ ]:
from pinatubo_steps.pha_comparison import compare_pick_csvs

COMPARISON_TOLERANCES = (0.05, 1.0)  # Seconds; reported separately.
COMPARISON_DATES = ['910603', '910610']  # Set None to report every pick date.
COMPARISON_OUTPUT = Path(SETTINGS['outdir']) / 'pha_cross_source'

comparison = compare_pick_csvs(
    SETTINGS['individual'], SETTINGS['monthly'],
    outdir=COMPARISON_OUTPUT,
    tolerances=COMPARISON_TOLERANCES,
    dates=COMPARISON_DATES,
)
print('Validation reports:', COMPARISON_OUTPUT)


In [ ]:
display(comparison['summary'])
display(comparison['timing_summary'])
print('Invalid source records (all dates):', len(comparison['invalid']))
display(comparison['invalid'].head(20))
print('Date discrepancies:')
display(comparison['date_audit'].loc[comparison['date_audit']['date_mismatch']].head(30))
print('Ambiguous candidate pairs:')
display(comparison['ambiguous_pairs'].head(30))
print('Unmatched picks by tolerance:')
display(comparison['unmatched'].head(30))
unmatched_ids = set(comparison['unmatched']['comparison_row_id'])
print('Nearest same-identity candidates for unmatched rows (diagnostic only):')
display(comparison['nearest_candidates'].loc[
    comparison['nearest_candidates']['comparison_row_id'].isin(unmatched_ids)
].head(30))
